# Controlled V013: improve diversity in the deep half of the ensemble

The previous fixed five-member average achieved official MAE **0.18025**. The strongest recent validated gain came from averaging different error patterns. This screen changes the deep half, holding features, grouped A split, full 200-epoch schedule, optimizer, preprocessing and the four compact-family members fixed.

1. `deep_seed03`: original deep architecture and loss, seed 20261103 instead of 20261102.
2. `deep_curve`: original deep architecture and seed, with the existing 0.1 curve-consistency auxiliary loss.

Three mixtures and their exact weights are frozen in `research/protocol_v013.json` before any new result. Promotion requires at least 0.003 A MAE improvement and a negative whole-breath bootstrap upper95 bound. Only the lowest eligible mixture can enter a new, cold-fitted E grouped confirmation. No weight search on E or official scores. A has been used repeatedly; confirmation is resampling evidence. Modeling uses approved Gym inputs only. Default execution is inert.

Reference official CSV, previous models, immutable sources and logs remain preserved. Synthetic resume checks run on the actual GPU before assignment training. Epoch logs expose progress independently of notebook output.


In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import copy, gc, hashlib, json, time, uuid, fcntl
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd

V013_PROTOCOL_SHA='6a32108a623a81b461e8b187784aa5bcf52a032d1bdd6c9a8fddb1cd77bc7cc1'
V013_NOTEBOOK='controlled_experiments_v013.ipynb'
V013_OUT='outputs/controlled_experiments_v013'
V013_TRAIN_SHA='2dc30dc5c8e603c8c68200830eab7ed8c7af42c823f8ae68b0beb663eea817d7'
V013_DTYPES=dict(id='int64',breath_id='int32',R='int8',C='int8',time_step='float32',u_in='float32',u_out='int8',pressure='float32')

def v013_sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def v013_ah(a):return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()
def v013_sig(v):return hashlib.sha256(json.dumps(v,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()
def v013_code(p):
    n=json.loads(Path(p).read_text());return hashlib.sha256('\n'.join(''.join(c['source']) for c in n['cells'] if c['cell_type']=='code').encode()).hexdigest()
def v013_require(c,message):
    if not bool(c):raise ValueError(message)
def v013_json(p,v):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);t=p.with_name(p.name+'.tmp-'+uuid.uuid4().hex)
    t.write_text(json.dumps(v,indent=2,allow_nan=False)+'\n');os.replace(t,p)
def v013_array(p,a):
    p=Path(p);t=p.with_name(p.name+'.tmp-'+uuid.uuid4().hex)
    with t.open('wb') as f:np.save(f,a,allow_pickle=False)
    os.replace(t,p)
def v013_local(root,value):
    marker='/ventilator-pressure-prediction/';s=str(value);p=(root/(s.split(marker,1)[1] if marker in s else s)).resolve()
    v013_require(p.is_relative_to(root),'Path outside project');return p
def v013_event(out,name,**kw):
    with (out/'events.jsonl').open('a') as f:f.write(json.dumps(dict(event=name,time_utc=datetime.now(timezone.utc).isoformat(),**kw))+'\n')
def v013_factory(spec,n_features,prep):return V5PressureRNN(spec,n_features)
v013_factory.__module__='controlled_experiments_v013'

def v013_blend(predictions,weights,rows):
    v013_require(all(w>0 for w in weights.values()) and abs(sum(weights.values())-1)<1e-12,'Invalid fixed weights')
    blend=np.zeros(rows,dtype=np.float64)
    for name,w in weights.items():
        a=predictions[name];v013_require(a.dtype==np.float64 and a.shape==(rows,) and np.isfinite(a).all(),'Prediction shape/dtype/finiteness changed')
        blend+=w*a
    return blend

def v013_interval(new_error,old_error,counts,seed=20261301):
    v013_require(len(new_error)==len(old_error)==int(counts.sum()) and (counts>0).all(),'Bootstrap alignment changed')
    sums=np.add.reduceat(new_error-old_error,np.r_[0,np.cumsum(counts)[:-1]])
    rng=np.random.default_rng(seed);draws=[]
    for _ in range(10):
        take=rng.integers(0,len(counts),size=(100,len(counts)));draws.extend(sums[take].sum(1)/counts[take].sum(1))
    lo,hi=np.quantile(draws,[.025,.975])
    return dict(difference=float((new_error-old_error).mean()),low=float(lo),high=float(hi),unit='whole breath',point_weighted=True,draws=1000,seed=seed)

def v013_diagnostics(out,name,prediction,truth,mask,rc):
    error=np.abs(prediction-truth);signed=prediction-truth
    rcs=np.repeat(rc,80,axis=0)[mask.ravel()]
    step=np.broadcast_to(np.arange(80),mask.shape)[mask]
    phase=np.minimum((5*np.arange(80)[None,:]/np.maximum(mask.sum(1)[:,None],1)).astype(int),4)[mask]
    t=pd.DataFrame(dict(R=rcs[:,0],C=rcs[:,1],step=step,phase=phase,absolute_error=error,signed_error=signed))
    for keys in [['R','C'],['step'],['phase']]:
        a=t.groupby(keys).agg(points=('absolute_error','size'),mae=('absolute_error','mean'),total_absolute_error=('absolute_error','sum'),bias=('signed_error','mean')).reset_index()
        a['fraction_total_error']=a.total_absolute_error/error.sum();a.to_csv(out/(name+'_by_'+'_'.join(keys)+'.csv'),index=False)
    return dict(mae=float(error.mean()),median_absolute_error=float(np.median(error)),p90_absolute_error=float(np.quantile(error,.9)))

def run_screen_v013(root='.',resume=False):
    root=Path(root).resolve();out=root/V013_OUT;out.mkdir(parents=True,exist_ok=True)
    locks=[];began=time.monotonic()
    try:
        for path in [root/'outputs/.v011_gpu.lock',out/'.run.lock']:
            f=path.open('a+');fcntl.flock(f,fcntl.LOCK_EX|fcntl.LOCK_NB);locks.append(f)
        v013_require(v013_sha(root/'research/protocol_v013.json')==V013_PROTOCOL_SHA,'Frozen protocol changed')
        plan=json.loads((root/'research/protocol_v013.json').read_text())
        for p,h in plan['inputs'].items():v013_require(v013_sha(root/p)==h,'Frozen input changed: '+p)
        v013_require(v013_sha(root/'data/train.csv')==V013_TRAIN_SHA,'Approved train changed')
        v013_require(v013_sha(root/plan['preserved_official_CSV'])==plan['preserved_official_CSV_sha256'],'Accepted official CSV changed')
        for n,h in plan['component_code_hashes'].items():
            v013_require(v013_code(root/'research'/n)==h,'Component code changed: '+n)
            book=json.loads((root/'research'/n).read_text())
            for i,c in enumerate(book['cells']):
                if c['cell_type']=='code':exec(compile(''.join(c['source']),n+':'+str(i),'exec'),globals())
        torch.set_num_threads(4);v013_require(torch.cuda.is_available(),'Approved CUDA GPU required')
        seed_sequence(20261102)
        source=v013_code(root/V013_NOTEBOOK)
        expected=os.environ.get('PA3_V013_SOURCE_SHA256');v013_require(source==expected,'Screen source differs from reviewed pin')
        fingerprints=dict(plan['component_code_hashes'],**{V013_NOTEBOOK:source,'protocol_v013.json':V013_PROTOCOL_SHA})
        design=dict(plan_sha256=V013_PROTOCOL_SHA,source_code_sha256=source,components=fingerprints,runtime=dict(torch=str(torch.__version__),cuda=torch.version.cuda,numpy=np.__version__,pandas=pd.__version__),strict_determinism=torch.are_deterministic_algorithms_enabled())
        v013_require(design['strict_determinism'],'Strict determinism required')
        if (out/'design.json').exists():v013_require(resume and json.loads((out/'design.json').read_text())==design,'Changed design or missing explicit resume')
        else:v013_json(out/'design.json',design)
        if not (out/'course_proofs.json').exists():
            proofs={}
            for arm,settings in plan['arms'].items():
                proofs[arm]=v8_training_selfcheck(v013_factory,arm=arm,loss_kind=settings['loss_kind'],source_fingerprints=fingerprints,device='cuda',feature_names=feature_names_v005('combined'),n_features=47)
            v013_json(out/'course_proofs.json',proofs)
        v013_event(out,'environment',pod=os.environ.get('HOSTNAME'),gpu=torch.cuda.get_device_name(0),design_sha256=v013_sig(design))
        raw=pd.read_csv(root/'data/train.csv',dtype=V013_DTYPES);bids=_validate_feature_frame_v005(raw)
        y=raw.pressure.to_numpy(copy=True).reshape(-1,80);uo=raw.u_out.to_numpy(copy=True).reshape(-1,80);ids=raw.id.to_numpy(copy=True).reshape(-1,80);rc=raw[['R','C']].to_numpy()[::80]
        known=raw[['id','breath_id','R','C','time_step','u_in','u_out']].copy();del raw;gc.collect()
        with np.load(root/'outputs/improvements_v005/sequence/split_breath_ids.npz',allow_pickle=False) as z:
            groups={k:np.sort(pd.Index(bids).get_indexer(z[k])) for k in ['A_fit','A_validation']}
        fit=groups['A_fit'];val=groups['A_validation']
        v013_require(len(fit)==61114 and len(val)==6791 and not np.intersect1d(fit,val).size and np.array_equal(np.sort(np.r_[fit,val]),np.arange(67905)),'Original grouped A split changed')
        mask=uo[val]==0;truth=y[val][mask].astype(np.float64);rows=ids[val][mask];counts=mask.sum(1)
        v013_require(len(rows)==206098 and np.array_equal(rows,np.load(root/'outputs/improvements_v005/sequence/A_validation_row_ids.npy')) and np.array_equal(rows,np.load(root/'outputs/diversity_screen_v012/A_row_ids.npy')),'Validation row alignment changed')
        v013_array(out/'A_row_ids.npy',rows)
        x,names,feature_bids=make_sequence_features(known,'combined');del known;gc.collect()
        v013_require(x.shape==(67905,80,47) and np.array_equal(feature_bids,bids),'Feature schema/order changed')
        predictions={n:np.load(root/p,allow_pickle=False) for n,p in plan['reference_prediction_files'].items()}
        reference=v013_blend(predictions,plan['reference_weights'],len(rows))
        v013_require(np.allclose(reference,np.load(root/'outputs/diversity_screen_v012/A_deep_half_compact_curve_memory_offset_predictions.npy'),rtol=0,atol=1e-12) and abs(np.abs(reference-truth).mean()-plan['reference_A_MAE'])<1e-12,'Current best reference changed')
        v013_diagnostics(out,'reference',reference,truth,mask,rc[val])
        # Training-only scaling and targets are handled independently for each arm
        # by the unchanged, source-pinned V008 trainer. No validation breath enters fit.
        records={}
        for arm,settings in plan['arms'].items():
            stage=out/arm;stage.mkdir(exist_ok=True);checkpoint=stage/'model.pt';rp=stage/'record.json'
            identity=dict(design_sha256=v013_sig(design),settings=settings,model_spec=plan['model_spec'],train_cfg=plan['train_cfg'],fit_group_sha256=v013_ah(bids[fit]),val_group_sha256=v013_ah(bids[val]),row_sha256=v013_ah(rows))
            if rp.exists():
                r=json.loads(rp.read_text());v013_require(resume and r['identity']==identity,'Completed arm identity changed')
                for p,h in [(r['checkpoint_file'],r['checkpoint_sha256']),(r['report_file'],r['report_sha256']),(r['prediction_file'],r['prediction_file_sha256'])]:v013_require(v013_sha(v013_local(root,p))==h,'Completed arm artifact changed')
            else:
                v013_event(out,'arm_started',arm=arm,seed=settings['seed'],loss=settings['loss_kind'])
                report=fit_validate_v008(x[fit],y[fit],uo[fit],x[val],y[val],uo[val],plan['model_spec'],plan['train_cfg'],settings['seed'],checkpoint,epoch_log_path=stage/'epochs.jsonl',feature_names=names,device='cuda',resume=resume and _last_path(checkpoint).exists(),model_factory=v013_factory,loss_kind=settings['loss_kind'],arm=arm,source_fingerprints=fingerprints)
                v013_require(report['completed'] and report['executed_epochs']==200,'Full A budget incomplete')
                v013_json(stage/'report.json',report)
                prediction=predict_checkpoint_v008(checkpoint,x[val],device='cuda',batch_size=512,model_factory=v013_factory,expected_arm=arm,expected_loss_kind=settings['loss_kind'],source_fingerprints=fingerprints)[mask].astype(np.float64)
                v013_require(abs(np.abs(prediction-truth).mean()-report['validation_mae'])<1e-6,'Checkpoint metric mismatch')
                pp=stage/'A_predictions.npy';v013_array(pp,prediction)
                r=dict(identity=identity,checkpoint_file=str(checkpoint.relative_to(root)),checkpoint_sha256=v013_sha(checkpoint),report_file=str((stage/'report.json').relative_to(root)),report_sha256=v013_sha(stage/'report.json'),prediction_file=str(pp.relative_to(root)),prediction_file_sha256=v013_sha(pp),prediction_array_sha256=v013_ah(prediction),best_epoch=report['best_epoch'],selected_lr_schedule=report['selected_lr_schedule'],individual_A_MAE=report['validation_mae'],training_eval_mae=report['training_eval_mae'],elapsed_seconds=report['elapsed_seconds'],seed=settings['seed'],loss_kind=settings['loss_kind'],training_breaths=len(fit),validation_breaths=len(val),parameter_count=report['parameter_count'])
                v013_json(rp,r)
            prediction=np.load(v013_local(root,r['prediction_file']),allow_pickle=False)
            v013_require(v013_ah(prediction)==r['prediction_array_sha256'],'Prediction array changed')
            predictions[arm]=prediction;records[arm]=r;v013_event(out,'arm_complete',arm=arm,individual_A_MAE=r['individual_A_MAE'])
        candidates=[];old_error=np.abs(reference-truth)
        for name,weights in plan['candidate_weights'].items():
            p=v013_blend(predictions,weights,len(rows));metric=v013_diagnostics(out,name,p,truth,mask,rc[val]);error=np.abs(p-truth)
            interval=v013_interval(error,old_error,counts,plan['bootstrap_seed']);gain=float(old_error.mean()-error.mean())
            v013_array(out/(name+'_A_predictions.npy'),p)
            candidates.append(dict(name=name,weights=weights,**metric,gain=gain,interval=interval,eligible=bool(gain>=plan['minimum_A_gain'] and interval['high']<0),prediction_file=str((out/(name+'_A_predictions.npy')).relative_to(root)),prediction_file_sha256=v013_sha(out/(name+'_A_predictions.npy'))))
        eligible=[r for r in candidates if r['eligible']];winner=min(eligible,key=lambda r:(r['mae'],r['name'])) if eligible else None
        v013_require(v013_code(root/V013_NOTEBOOK)==source,'Source changed during screen')
        for p,h in plan['inputs'].items():v013_require(v013_sha(root/p)==h,'Input changed during screen: '+p)
        v013_require(v013_sha(root/plan['preserved_official_CSV'])==plan['preserved_official_CSV_sha256'],'Accepted CSV changed')
        result=dict(status='complete',design_sha256=v013_sig(design),protocol_sha256=V013_PROTOCOL_SHA,source_code_sha256=source,reference_A_MAE=float(old_error.mean()),candidates=candidates,selected_for_confirmation=winner,records=records,confirmation_not_performed=True,final_fit_not_performed=True,submission_not_performed=True,official_best_MAE_preserved=.18025,elapsed_seconds=time.monotonic()-began,completed_utc=datetime.now(timezone.utc).isoformat())
        v013_json(out/'screen_summary.json',result);v013_event(out,'screen_complete',winner=winner)
        return result
    finally:
        for f in reversed(locks):fcntl.flock(f,fcntl.LOCK_UN);f.close()

if os.environ.get('PA3_RUN_V013','0')=='1':
    V013_RESULT=run_screen_v013(Path.cwd(),resume=os.environ.get('PA3_RESUME_V013','0')=='1')
    print(json.dumps({k:v for k,v in V013_RESULT.items() if k!='records'},indent=2))
